# Train + Hyperparameter Tuning (MLflow)

This notebook:
- loads the dataset
- logs each trial as an MLflow run (params, metrics, model artifact)
- finds the best run and registers the model into MLflow Model Registry

### Objective

This notebook aims to train and evaluate ML models on the Iris dataset, and store results in **Google Cloud Storage**.

It uses the following Google Cloud ML services and resources:

- Vertex AI Workbench
- Cloud Storage

The steps performed include:

- Setting up environment and dependencies
- Loading and augmenting the Iris dataset
- Training ML models
- Saving artifacts (models and metrics) to GCS
- Running inference on evaluation data


### Dataset

We use the classic **Iris dataset**, which contains 150 rows of flower measurements across three species (`setosa`, `versicolor`, `virginica`).  
The dataset is available in the resources that are provided for this assignment: [Iris dataset](https://github.com/IITMBSMLOps/ga_resources/tree/week_1/data)  

For this assignment, the dataset is stored in a Google Cloud Storage bucket under the `data/` folder.


### Costs 

Billable components of Google Cloud:

* Vertex AI
* Cloud Storage

Learn about [Vertex AI pricing](https://cloud.google.com/vertex-ai/pricing)  
and [Cloud Storage pricing](https://cloud.google.com/storage/pricing).  
Use the [Pricing Calculator](https://cloud.google.com/products/calculator/) to estimate costs based on usage.

## Installation

Install the following packages required to execute this notebook. 

In [ ]:
! pip3 install --upgrade --quiet google-cloud-aiplatform

### Setting up Google Cloud project

**The following steps are required, regardless of your notebook environment.**

1. [Select or create a Google Cloud project](https://console.cloud.google.com/cloud-resource-manager). New accounts come with $300 free credits.
2. [Make sure that billing is enabled for your project](https://cloud.google.com/billing/docs/how-to/modify-project).
3. [Enable the Vertex AI API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).
4. Also enable [Cloud Storage API](https://console.cloud.google.com/flows/enableapi?apiid=storage.googleapis.com).
5. If you are running this notebook locally, install the [Cloud SDK](https://cloud.google.com/sdk).

#### Set your project ID

**If you don't know your project ID**, try the following:
* Run `gcloud config list`.
* See the support page: [Locate the project ID](https://support.google.com/googleapi/answer/7014113)

In [ ]:
! gcloud config list
# ! gcloud projects list

[compute]
region = us-central1
[core]
account = 898498043475-compute@developer.gserviceaccount.com
disable_usage_reporting = True
project = mlops-473405
[dataproc]
region = us-central1

Your active configuration is: [default]


In [1]:
PROJECT_ID = "mlops-473405"  # @param {type:"string"}

# Set the project id
! gcloud config set project {PROJECT_ID}

Updated property [core/project].


#### Region

You can also change the `REGION` variable used by Vertex AI. Learn more about [Vertex AI regions](https://cloud.google.com/vertex-ai/docs/general/locations).

In [2]:
REGION = "us-central1"  # @param {type: "string"}

### Create a Cloud Storage bucket

Create a storage bucket to store intermediate artifacts such as datasets.

- *{Note to notebook author: For any user-provided strings that need to be unique (like bucket names or model ID's), append "-unique" to the end so proper testing can occur}*

In [3]:
BUCKET_NAME = "week5_bucket"
BUCKET_URI = f"gs://{BUCKET_NAME}"  # @param {type:"string"}
FILE_NAME = "data/iris.csv"

In [4]:
! gsutil mb -l {REGION} -p {PROJECT_ID} {BUCKET_URI}

Creating gs://week5_bucket/...


### Import libraries

In [5]:
from google.cloud import aiplatform

### Initialize Vertex AI SDK for Python

Initialize the Vertex AI SDK for Python for your project.

In [6]:
aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=BUCKET_URI)

In [23]:
! pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [optuna]2m1/2 [optuna]

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [4]:
import os
import pandas as pd
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [8]:
# Update host and port if running remotely.
mlflow.set_tracking_uri("http://127.0.0.1:8100")

EXPERIMENT_1 = "RF_depth3"
EXPERIMENT_2 = "RF_depth6"
MODEL_NAME = "iris_random_forest_model"
DATA_PATH = "data/iris.csv

print("MLFLOW_TRACKING_URI:", mlflow.get_tracking_uri())

MLFLOW_TRACKING_URI: http://127.0.0.1:8100


In [6]:
# --- Remove model logging from DVC ---
# This step ensures models are tracked only by MLflow, not DVC.

!dvc remove models/model.joblib.dvc --force || echo "No model tracked in DVC."
!git rm models/model.joblib.dvc --force || echo "File not found in git."
!git commit -m "Removed model logging from DVC (migrated to MLflow)" || echo "Commit skipped."

/bin/bash: line 1: dvc: command not found
No model tracked in DVC.
fatal: not a git repository (or any parent up to mount point /home)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
File not found in git.
fatal: not a git repository (or any parent up to mount point /home)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
Commit skipped.


In [9]:
df = pd.read_csv(DATA_PATH)
# if using Feast, you'd fetch historical features earlier
X = df[["sepal_length","sepal_width","petal_length","petal_width"]]
y = df["species"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [10]:
# --- Experiment 1: Random Forest (depth = 3) ---
mlflow.set_experiment(EXPERIMENT_1)

n_estimators = 100
max_depth = 3

with mlflow.start_run(run_name="RF_depth3"):
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)

    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_metric("accuracy", acc)

    mlflow.sklearn.log_model(model, artifact_path="model")
    print(f"Model v1 logged with accuracy: {acc:.4f}")


2025/10/26 04:27:06 INFO mlflow.tracking.fluent: Experiment with name 'RF_depth3' does not exist. Creating a new experiment.
2025/10/26 04:27:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/10/26 04:27:18 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Model v1 logged with accuracy: 1.0000
🏃 View run RF_depth3 at: http://127.0.0.1:8100/#/experiments/761019764260581469/runs/8fe06db81e2c423eba4593f7f05851c9
🧪 View experiment at: http://127.0.0.1:8100/#/experiments/761019764260581469


In [15]:
# --- Experiment 2: Random Forest (depth = 6) ---
mlflow.set_experiment(EXPERIMENT_2)

n_estimators = 150
max_depth = 6

with mlflow.start_run(run_name="RF_depth6"):
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)

    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_metric("accuracy", acc)

    mlflow.sklearn.log_model(model, artifact_path="model")
    print(f"Model v2 logged with accuracy: {acc:.4f}")


2025/10/26 04:51:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/10/26 04:51:22 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Model v2 logged with accuracy: 1.0000
🏃 View run RF_depth6 at: http://127.0.0.1:8100/#/experiments/527143898048398913/runs/0b44f692235543e0ab83e3afcfa1e23f
🧪 View experiment at: http://127.0.0.1:8100/#/experiments/527143898048398913


In [12]:
# --- Register the best performing model ---
client = MlflowClient()

# Get both experiments
exp1 = client.get_experiment_by_name(EXPERIMENT_1)
exp2 = client.get_experiment_by_name(EXPERIMENT_2)

# Search and find best run from both experiments
runs_1 = client.search_runs([exp1.experiment_id], order_by=["metrics.accuracy DESC"], max_results=1)
runs_2 = client.search_runs([exp2.experiment_id], order_by=["metrics.accuracy DESC"], max_results=1)

best_run = max([runs_1[0], runs_2[0]], key=lambda r: r.data.metrics["accuracy"])
best_run_id = best_run.info.run_id
best_acc = best_run.data.metrics["accuracy"]

print(f"Best Run ID: {best_run_id}, Accuracy: {best_acc:.4f}")

# Register model in MLflow Model Registry
model_uri = f"runs:/{best_run_id}/model"
try:
    client.create_registered_model(MODEL_NAME)
except Exception:
    pass

model_version = client.create_model_version(name=MODEL_NAME, source=model_uri, run_id=best_run_id)
client.transition_model_version_stage(
    name=MODEL_NAME, version=model_version.version, stage="Production", archive_existing_versions=True
)

print(f"Registered {MODEL_NAME} version {model_version.version} and promoted to Production")


2025/10/26 04:35:14 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: iris_random_forest_model, version 1


Best Run ID: 8fe06db81e2c423eba4593f7f05851c9, Accuracy: 1.0000
Registered iris_random_forest_model version 1 and promoted to Production


/var/tmp/ipykernel_7114/3471931439.py:26: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


In [13]:
# --- Evaluate best model from registry ---
model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}/Production")

preds = model.predict(X_test)
acc = accuracy_score(y_test, preds)
print("Evaluation Accuracy:", acc)
print("Classification Report:\n", classification_report(y_test, preds))


Evaluation Accuracy: 1.0
Classification Report:
               precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00         9
   virginica       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



## Cleaning up

To clean up all Google Cloud resources used in this project, you can [delete the Google Cloud
project](https://cloud.google.com/resource-manager/docs/creating-managing-projects#shutting_down_projects) you used for the tutorial.

Otherwise, you can delete the individual resources you created in this tutorial:

{TODO: Include commands to delete individual resources below}

In [ ]:
import os

# Delete endpoint resource
# e.g. `endpoint.delete()`

# Delete model resource
# e.g. `model.delete()`

# Delete Cloud Storage objects that were created
# delete_bucket = False
# if delete_bucket or os.getenv("IS_TESTING"):
#     ! gsutil -m rm -r $BUCKET_URI